# Module 09: High-Performance Persistence & Parallel Computing with Joblib
## Notebook 01: Machine Learning Model Persistence and Compression

`joblib` is a set of tools to provide lightweight pipelining in Python. In the machine learning ecosystem, `joblib` is especially renowned as the default persistence engine for Scikit-Learn. It is specifically optimized to serialize arbitrary Python objects that contain large **NumPy array buffers** efficiently.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand why `joblib` outperforms standard `pickle` on NumPy-heavy machine learning models.
2. Persist and restore machine learning estimators using `joblib.dump()` and `joblib.load()`.
3. Compare built-in compression algorithms: **Zlib**, **Gzip**, **Bzip2**, and **LZMA**.
4. Benchmark disk storage footprint vs. serialization latency.
5. **Advanced:** Serialize models directly into in-memory byte buffers (`io.BytesIO`) for cloud and microservice deployment.
6. **Advanced:** Build a production **Model Registry Envelope** bundling trained pipelines with metadata, feature schemas, and performance benchmarks.

In [1]:
import os
import io
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

data_dir = "data_files" if os.path.exists("data_files") else "../data_files"
print(f"Joblib Version: {joblib.__version__}")

Joblib Version: 1.6.0


### 1. Joblib vs. Standard Pickle for Machine Learning
Why use `joblib` over `pickle`?
- **NumPy Array Optimization:** Machine learning models (Random Forests, Gradient Boosting Trees, SVMs, PCA) store massive parameter arrays (tree split thresholds, support vectors, covariance matrices).
- `joblib.dump()` stores contiguous NumPy memory blocks with optimal alignment, avoiding the overhead of serializing individual Python objects.

In [2]:
# Train a substantial Random Forest Classifier (100 trees, 15 max depth)
X, y = make_classification(n_samples=2500, n_features=25, n_informative=18, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
rf_model.fit(X, y)
print(f"Trained Random Forest with {len(rf_model.estimators_)} trees across {X.shape[1]} features.")

# Baseline persistence with joblib.dump
base_path = "rf_model.joblib"
joblib.dump(rf_model, base_path)
print(f"Persisted to {base_path}: {os.path.getsize(base_path) / 1024:.1f} KB")

# Load and verify inference predictions
loaded_rf = joblib.load(base_path)
test_sample = X[:5]
np.testing.assert_allclose(rf_model.predict_proba(test_sample), loaded_rf.predict_proba(test_sample))
print("SUCCESS: Loaded model outputs identical prediction probabilities!")

if os.path.exists(base_path):
    os.remove(base_path)

Trained Random Forest with 100 trees across 25 features.
Persisted to rf_model.joblib: 3954.4 KB
SUCCESS: Loaded model outputs identical prediction probabilities!


### 2. Compression Algorithms & Trade-off Analysis
`joblib.dump()` supports built-in compression via the `compress` parameter:
- **`compress=0`:** Uncompressed (fastest write/read, largest file size).
- **`compress=3` (or `('zlib', 3)`):** Default zlib compression (balanced speed and compression ratio).
- **`compress=('gzip', 3)`:** Gzip compression (widely supported by cloud infrastructure).
- **`compress=('bz2', 3)`:** Bzip2 (higher compression ratio, slower write speed).
- **`compress=('lzma', 3)`:** LZMA (highest compression ratio, most CPU-intensive).

In [3]:
# Benchmark compression algorithms on the Random Forest model
algorithms = [
    ("Uncompressed", 0),
    ("Zlib (level 3)", ("zlib", 3)),
    ("Gzip (level 3)", ("gzip", 3)),
    ("Bzip2 (level 3)", ("bz2", 3)),
    ("LZMA (level 3)", ("lzma", 3))
]

print(f"{'Algorithm':<18} | {'File Size (KB)':<14} | {'Save Time (ms)':<15} | {'Load Time (ms)':<15}")
print("-" * 70)

bench_results = []
for name, comp in algorithms:
    fname = f"model_bench_{name.split()[0].lower()}.joblib"

    # Measure Save Latency
    t0 = time.time()
    joblib.dump(rf_model, fname, compress=comp)
    save_ms = (time.time() - t0) * 1000

    file_kb = os.path.getsize(fname) / 1024.0

    # Measure Load Latency
    t0 = time.time()
    _ = joblib.load(fname)
    load_ms = (time.time() - t0) * 1000

    print(f"{name:<18} | {file_kb:<14.1f} | {save_ms:<15.2f} | {load_ms:<15.2f}")
    bench_results.append((name, file_kb, save_ms, load_ms))

    # Clean up benchmark file
    if os.path.exists(fname):
        os.remove(fname)

Algorithm          | File Size (KB) | Save Time (ms)  | Load Time (ms) 
----------------------------------------------------------------------
Uncompressed       | 3954.4         | 29.08           | 22.65          
Zlib (level 3)     | 929.0          | 81.13           | 35.07          
Gzip (level 3)     | 929.0          | 79.57           | 32.93          
Bzip2 (level 3)    | 552.6          | 234.25          | 95.86          
LZMA (level 3)     | 490.8          | 315.62          | 46.85          


### 3. Complex Application 1: In-Memory Serialization with `io.BytesIO`
In serverless microservices (AWS Lambda, Google Cloud Functions) and streaming architectures, saving models to the physical filesystem is often prohibited or undesirable.
`joblib` supports serializing directly to in-memory byte buffers via Python's `io.BytesIO`:

In [4]:
# Serialize model into an in-memory byte buffer
buffer = io.BytesIO()
joblib.dump(rf_model, buffer, compress=3)

# Inspect in-memory buffer
raw_bytes = buffer.getvalue()
print(f"In-Memory Model Buffer Size: {len(raw_bytes) / 1024:.2f} KB")

# Reset stream position and restore model
buffer.seek(0)
restored_from_memory = joblib.load(buffer)

test_pred = restored_from_memory.predict(X[:3])
print(f"Predictions from In-Memory Restored Model: {test_pred.tolist()}")

In-Memory Model Buffer Size: 929.02 KB
Predictions from In-Memory Restored Model: [1, 1, 1]


### 4. Complex Application 2: Production Model Registry Envelope
In enterprise ML, saving *just* the model object is an anti-pattern:
- What features did the model expect?
- What was the training timestamp, Git commit SHA, and test accuracy?
- What versions of Scikit-Learn and Joblib produced this artifact?
**Best Practice:** Package the model inside a versioned **Model Envelope**:

In [5]:
import sklearn

# Construct Production Model Registry Envelope
feature_columns = [f"signal_feature_{i+1:02d}" for i in range(X.shape[1])]

model_envelope = {
    "model": rf_model,
    "metadata": {
        "model_type": "RandomForestClassifier",
        "version": "2.4.1",
        "author": "MLOps Automated Training Pipeline",
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "sklearn_version": sklearn.__version__,
        "joblib_version": joblib.__version__
    },
    "schema": {
        "features": feature_columns,
        "num_features": len(feature_columns),
        "target_classes": [0, 1]
    },
    "metrics": {
        "train_accuracy": float(rf_model.score(X, y)),
        "n_samples": X.shape[0]
    }
}

envelope_path = "production_rf_envelope.joblib"
joblib.dump(model_envelope, envelope_path, compress=("zlib", 3))

print(f"Saved Production Envelope to {envelope_path} ({os.path.getsize(envelope_path) / 1024:.1f} KB)")

# Load and validate schema before inference
deployed_package = joblib.load(envelope_path)
print(f"\n--- LOADED MODEL ENVELOPE METADATA ---")
for k, v in deployed_package["metadata"].items():
    print(f"  {k:<16}: {v}")

print(f"  {'Train Accuracy':<16}: {deployed_package['metrics']['train_accuracy'] * 100:.2f}%")

if os.path.exists(envelope_path):
    os.remove(envelope_path)

Saved Production Envelope to production_rf_envelope.joblib (929.5 KB)

--- LOADED MODEL ENVELOPE METADATA ---
  model_type      : RandomForestClassifier
  version         : 2.4.1
  author          : MLOps Automated Training Pipeline
  created_at      : 2026-09-25T18:59:47Z
  sklearn_version : 1.9.1
  joblib_version  : 1.6.0
  Train Accuracy  : 100.00%
